<a href="https://colab.research.google.com/github/anik2644/44_react_lab/blob/main/beneficiary_fetcher_colab_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SNSOP Beneficiary Info Fetcher - Colab Edition

This notebook connects to the SQL Server database and provides functions for beneficiary search, statistics, and optional LLM-assisted chat.

In [ ]:
# 1. Install Dependencies
import sys
import subprocess

def install_packages():
    packages = [
        "pymssql==2.3.13",
        "SQLAlchemy==2.0.49",
        "pydantic==2.13.3",
        "pydantic-settings==2.14.0",
        "python-dotenv==1.2.2",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

install_packages()

import json
import re
from datetime import datetime
from enum import IntEnum
from getpass import getpass
from typing import Optional

from pydantic import BaseModel
from sqlalchemy import Boolean, DateTime, Integer, String, URL, create_engine, func, or_
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, sessionmaker

# ---------------------------------------------------------------------
# Database setup
# ---------------------------------------------------------------------

print("Enter your SQL Server connection details.")
DB_HOST = input("DB_HOST: ").strip()
DB_PORT_RAW = input("DB_PORT [1433]: ").strip()
DB_PORT = int(DB_PORT_RAW or "1433")
DB_NAME = input("DB_NAME: ").strip()
DB_USER = input("DB_USER: ").strip()
DB_PASSWORD = getpass("DB_PASSWORD: ")

DB_URL = URL.create(
    "mssql+pymssql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(DB_URL, pool_pre_ping=True)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

class Base(DeclarativeBase):
    pass

# ---------------------------------------------------------------------
# Enums and display values
# ---------------------------------------------------------------------

class GenderEnum(IntEnum):
    MALE = 0
    FEMALE = 1

class LegalStatusEnum(IntEnum):
    HOST = 0
    RETURNEE = 1
    REFUGEE = 2
    IDP = 3

class MaritalStatusEnum(IntEnum):
    SINGLE = 0
    MARRIED = 1
    WIDOW = 2
    SEPARATED = 3
    DIVORCE = 4

class SelectionCriteriaEnum(IntEnum):
    LIPW = 0
    DIS = 1
    ALL = 2

GENDER_DISPLAY = {GenderEnum.MALE: "M", GenderEnum.FEMALE: "F"}
LEGAL_STATUS_DISPLAY = {
    LegalStatusEnum.HOST: "Host",
    LegalStatusEnum.RETURNEE: "Returnee",
    LegalStatusEnum.REFUGEE: "Refugee",
    LegalStatusEnum.IDP: "Idp",
}
MARITAL_STATUS_DISPLAY = {
    MaritalStatusEnum.SINGLE: "Single",
    MaritalStatusEnum.MARRIED: "Married",
    MaritalStatusEnum.WIDOW: "Widow",
    MaritalStatusEnum.SEPARATED: "Separated",
    MaritalStatusEnum.DIVORCE: "Divorce",
}
SELECTION_CRITERIA_DISPLAY = {
    SelectionCriteriaEnum.LIPW: "Component 1 (Public Works)",
    SelectionCriteriaEnum.DIS: "Component 1 (Direct Income Support)",
    SelectionCriteriaEnum.ALL: "All",
}

# ---------------------------------------------------------------------
# Database model
# ---------------------------------------------------------------------

class Beneficiary(Base):
    __tablename__ = "p2_beneficiary"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    household_number: Mapped[str] = mapped_column(String(36), unique=True, nullable=False)
    first_name: Mapped[str] = mapped_column(String(32), nullable=False)
    middle_name: Mapped[str] = mapped_column(String(32), nullable=False)
    last_name: Mapped[str] = mapped_column(String(32), nullable=False)
    gender: Mapped[int] = mapped_column(Integer, nullable=False)
    age: Mapped[int] = mapped_column(Integer, nullable=False)
    legal_status: Mapped[int] = mapped_column(Integer, nullable=False)
    marital_status: Mapped[int] = mapped_column(Integer, nullable=False)
    selection_criteria: Mapped[int] = mapped_column(Integer, nullable=False)
    phone_no: Mapped[str | None] = mapped_column(String(14))
    can_read_write: Mapped[bool] = mapped_column(Boolean, nullable=False)
    has_mobile_wallet: Mapped[bool | None] = mapped_column(Boolean)
    afis_status: Mapped[int | None] = mapped_column(Integer)
    id_card_generated: Mapped[bool | None] = mapped_column(Boolean)

def beneficiary_to_dict(b: Beneficiary) -> dict:
    row = {column.name: getattr(b, column.name) for column in b.__table__.columns}
    row["full_name"] = f"{b.first_name} {b.middle_name} {b.last_name}"
    row["gender_display"] = GENDER_DISPLAY.get(b.gender)
    row["legal_status_display"] = LEGAL_STATUS_DISPLAY.get(b.legal_status)
    row["marital_status_display"] = MARITAL_STATUS_DISPLAY.get(b.marital_status)
    row["selection_criteria_display"] = SELECTION_CRITERIA_DISPLAY.get(b.selection_criteria)
    return row

def json_print(data):
    print(json.dumps(data, default=str, indent=2))

# ---------------------------------------------------------------------
# Queries and Stats
# ---------------------------------------------------------------------

def stats_overview() -> dict:
    with SessionLocal() as db:
        total = db.query(func.count(Beneficiary.id)).scalar()
        avg_age = db.query(func.avg(Beneficiary.age)).scalar()
        with_wallet = db.query(func.count(Beneficiary.id)).filter(Beneficiary.has_mobile_wallet == True).scalar()
        return {
            "total_beneficiaries": total,
            "average_age": round(float(avg_age), 1) if avg_age else None,
            "with_mobile_wallet": with_wallet,
        }

def search_beneficiaries(text: str, limit: int = 10) -> list[dict]:
    with SessionLocal() as db:
        term = f"%{text}%"
        full_name = Beneficiary.first_name + " " + Beneficiary.middle_name + " " + Beneficiary.last_name
        items = db.query(Beneficiary).filter(or_(
            full_name.ilike(term),
            Beneficiary.household_number.ilike(term),
            Beneficiary.phone_no.ilike(term),
        )).limit(limit).all()
        return [beneficiary_to_dict(item) for item in items]

def answer_question(question: str):
    q = question.lower()
    if "overview" in q or "total" in q:
        return stats_overview()

    hh_match = re.search(r"household\s+([A-Za-z0-9_-]+)", q)
    if hh_match:
        with SessionLocal() as db:
            item = db.query(Beneficiary).filter(Beneficiary.household_number == hh_match.group(1)).first()
            return beneficiary_to_dict(item) if item else "Not found."

    return "Try asking for 'overview' or 'household [number]'."

# ---------------------------------------------------------------------
# Execution
# ---------------------------------------------------------------------

try:
    with engine.connect() as connection:
        print("\n[SUCCESS] Database connection established.")
        json_print(stats_overview())
except Exception as exc:
    print(f"\n[ERROR] Connection failed: {exc}")
